<a href="https://colab.research.google.com/github/CarterKatz/homework_02_rnn_text_experiment.ipynb/blob/main/homework_02_rnn_text_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 2 - RNNs with Pretrained Word Embeddings

Train one recurrent neural network using frozen pretrained word embeddings. Use the trained RNN for AG News classification, document similarity, and query-based retrieval.

Choose one framework: PyTorch or TensorFlow with Keras. You do not need to implement both. You may use a Simple RNN, GRU, or LSTM. A bidirectional model is allowed but not required. Transformers are outside the scope of this assignment.

Implementation details are intentionally open-ended. Briefly describe the choices you make.

## Learning goals

- Prepare token sequences for a recurrent neural network.
- Initialize an embedding layer with pretrained word vectors.
- Freeze the pretrained embedding layer.
- Train an RNN, GRU, or LSTM for text classification.
- Extract one vector for each document from the trained RNN.
- Use document vectors for similarity and search.
- Use a GPU when one is available.

# Using a GPU

Google Colab is recommended if you do not have access to a local GPU.

1. Open the notebook in Google Colab.
2. Open the Runtime menu.
3. Select Change runtime type.
4. Select a GPU hardware accelerator.
5. Restart the runtime if Colab requests it.
6. Run the device check before training.

The command below will fail or show no GPU if a GPU runtime is not active. Run only the device example for your selected framework. PyTorch users must move the model and every training batch to the selected device. Keras normally uses an available GPU automatically.

GPU reference: https://research.google.com/colaboratory/faq.html#gpu-availability

In [1]:
# Optional GPU check
!nvidia-smi

# PyTorch example
# import torch
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Selected device", device)

# TensorFlow example
import tensorflow as tf
print("Available GPUs", tf.config.list_physical_devices("GPU"))

Tue Sep 15 22:26:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Package setup

Run only the commands needed by your environment. Google Colab usually includes PyTorch and TensorFlow. Gensim may need to be installed. Do not import both deep-learning frameworks unless you want to.

In [2]:
%pip install gensim
# %pip install torch
%pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 66.6 MB/s eta 0:00:00


# The Experiment

1. Tokenize the training documents.
2. Build a vocabulary from the training text.
3. Convert documents into padded token sequences.
4. Load pretrained word vectors.
5. Initialize a frozen embedding layer.
6. Train a recurrent classifier.
7. Extract document vectors from the trained RNN.
8. Use the vectors for classification, similarity, and search.

The implementation details are open-ended. Record the choices you make.

# Load AG News

The source files have no header and contain label, title, and description. Labels 1 through 4 mean World, Sports, Business, and Sci/Tech. The official test data must remain separate from training data. The code below downloads both files and selects reproducible balanced subsets. You may reduce the training sample to 2,000 documents per category if runtime or memory is limited. Report the sample size you use.

In [3]:
import pandas as pd

TRAIN_URL = (
    "https://raw.githubusercontent.com/mhjabreel/"
    "CharCnn_Keras/master/data/ag_news_csv/train.csv"
)
TEST_URL = (
    "https://raw.githubusercontent.com/mhjabreel/"
    "CharCnn_Keras/master/data/ag_news_csv/test.csv"
)
LABEL_NAMES = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
RANDOM_STATE = 42

def load_ag_news(url):
    frame = pd.read_csv(url, header=None, names=["label", "title", "description"])
    frame["text"] = frame["title"].fillna("") + " " + frame["description"].fillna("")
    frame["label_name"] = frame["label"].map(LABEL_NAMES)
    frame["label_index"] = frame["label"] - 1
    return frame

train_all = load_ag_news(TRAIN_URL)
test_all = load_ag_news(TEST_URL)
train_per_category = 5000
test_per_category = 1000

train_df = (train_all.groupby("label", group_keys=False)
            .sample(n=train_per_category, random_state=RANDOM_STATE)
            .sample(frac=1, random_state=RANDOM_STATE)
            .reset_index(drop=True))
test_df = (test_all.groupby("label", group_keys=False)
           .sample(n=test_per_category, random_state=RANDOM_STATE)
           .sample(frac=1, random_state=RANDOM_STATE)
           .reset_index(drop=True))

train_df["doc_id"] = [f"train-{i:05d}" for i in range(len(train_df))]
test_df["doc_id"] = [f"test-{i:05d}" for i in range(len(test_df))]

print("Training shape", train_df.shape)
print("Test shape", test_df.shape)
print("Training counts\n", train_df["label_name"].value_counts())
print("Test counts\n", test_df["label_name"].value_counts())
display(train_df[["label_name", "text"]].head(4))

Training shape (20000, 7)
Test shape (4000, 7)
Training counts
 label_name
Business    5000
World       5000
Sports      5000
Sci/Tech    5000
Name: count, dtype: int64
Test counts
 label_name
World       1000
Sci/Tech    1000
Business    1000
Sports      1000
Name: count, dtype: int64


,label_name,text
0,Business,Wal-Mart Launches Rare Newspaper Ad Blitz (Reu...
1,World,Hirst restaurant sale makes 11m Fixtures and f...
2,Sports,Miami Heat Team Report - November 10 (Sports N...
3,World,Ukrainian Government Blamed for Poisoning Oppo...


# Prepare Token Sequences

Build the vocabulary from training text only. Lowercase and tokenize each document. Reserve indices for padding and unknown words. Truncate long sequences, pad short sequences, and apply the same vocabulary to test documents and search queries. A simple regex tokenizer is acceptable. A framework tokenizer is also acceptable if you explain it. Packed sequences are optional research for PyTorch students.

Recommended settings:

`MAX_VOCAB_SIZE = 20000`, `MAX_SEQUENCE_LENGTH = 60`, `PAD_INDEX = 0`, `UNK_INDEX = 1`, and `RANDOM_STATE = 42`.

References: [PyTorch padding](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html), [PyTorch packed sequences](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html), [Keras TextVectorization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization), and [Keras padding](https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences).

In [4]:
#https://www.geeksforgeeks.org/machine-learning/gated-recurrent-unit-networks/
#https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization#finalize_state
#https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences
#https://www.google.com/search?q=tf.keras.layers.TextVectorization+embedding+with+%22glove-wiki-gigaword-50%22&client=ubuntu-sn&hs=ZqH&sca_esv=0ad83d8dec62c36e&channel=fs&sxsrf=APpeQnti51lMlVrZt35ta_he-U_10ahS6Q%3A1789416375207&ei=cFCoarqhFu6uptQPrMvW6QM&biw=1040&bih=1293&uact=5&sclient=gws-wiz-serp&fbs=ABfTbFVyMZGZf1hfvX9uKjN_-G8c4u0nXx4bEIpwm1lnNH832SMIiTl3t-JZ4hGJOxPbHYSIu8Q64jU5EwQ-803VaKbd8XGNh2EAGT96nVa30badWeLcMOEWQcU9WGmTgQb8mz6mPncLV6BMXrUfQD5ftu4HKiVJOOiqhNYT8Keh_15lhVLohIhM8kHnw_tixWHDpdAHWCTimc1GO2znH-0EdfLD74c9fw&aep=10&ntc=1&mstk=AUtExfBG0wRmLXg8gYohrSiMuSkumSnj4pVXixXCC9XgQwxqzTPgUlxMVkdukuFs4dQ9pETHbpSvRY2gF9eVDg8o_U3CPW7RMbYvPi9hs8Ov1yd_B-XWbzEciNKgwkJ3peNYTj-gIIXcv1-2h5IJrUaghx_k9mKPy2am0po&aioh=3&csuir=1&udm=50&mtid=mlmoapvCIrCV0PEPn7vviQE&atvm=2
#Regular Expressions
import re
#Numerical operations
import numpy as np
import pandas as pd
#Machine learning stuff
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, TextVectorization, Embedding
from tensorflow.keras.optimizers import Adam
# I used Gemini's "explain code feature to help with notes"
MAX_VOCAB_SIZE = 20000 #20000 most frequent words will be analysed
MAX_SEQUENCE_LENGTH = 60 #All nput sequences will be padded or truncated to this fixed length.
PAD_INDEX = 0 # The integer value used for padding shorter sequences
UNK_INDEX = 1 # The integer value assigned to words not found in the vocabulary (unknown words)
TOKEN_PATTERN = r"\b[a-zA-Z][a-zA-Z']*\b"
RANDOM_STATE = 42

# build a training-only vocabulary, integer sequences, truncation, and padding.
# Keep PAD_INDEX and UNK_INDEX reserved, then encode test_tokens with the same mapping.
# Named padded arrays train_sequences and test_sequences.
# Prompt: "Explain Code" - I would iteratively use this prompt to proofread syntax
# The only code that this prompt added was the train_vector.adapt(train_df["text"])

train_Vector = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE, # Tokens represent words or ideas as "ngrams"
    standardize='lower_and_strip_punctuation', #Remove upper case and strip punctuation
    split='whitespace', # Divide words by spaces
    output_mode='int', #Output is integer sequences
    output_sequence_length=MAX_SEQUENCE_LENGTH
)
train_Vector.adapt(train_df["text"]) # "Adapt to the raw text"- Gemini
# The adapt method processes the train_df["text"] (raw training documents) to build the vocabulary.
#It learns the unique words and assigns them integer IDs. Importantly, it only uses the training data for this,
#preventing data leakage from the test set.
integer_train_sequences = train_Vector(train_df["text"])
train_sequences = tf.keras.utils.pad_sequences(
    sequences = integer_train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    dtype='int32',
    padding='pre',# Adds padding zeros at the beginning of sequences.
    truncating='pre',#Truncates longer sequences from the beginning.
    value=PAD_INDEX
)
integer_test_sequences = train_Vector(test_df["text"])
test_sequences = tf.keras.utils.pad_sequences(
    sequences = integer_test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    dtype='int32',
    padding='pre',# Adds padding zeros at the beginning of sequences.
    truncating='pre',#Truncates longer sequences from the beginning.
    value=PAD_INDEX
)

vocabulary = train_Vector.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary)))) #Creates a dictionary mapping each word in the vocabulary list
#to its corresponding integer index. This is a common utility for looking up word IDs.


# Pretrained Word Embeddings

GloVe is a pretrained static word-embedding model. It serves the same role as pretrained Word2Vec vectors in this experiment. Do not use the large Google News Word2Vec model. The embedding layer must be initialized from GloVe and remain frozen during training.

In [6]:
import gensim.downloader as api

word_vectors = api.load("glove-wiki-gigaword-50")
print("Embedding dimension", word_vectors.vector_size)

[==================================================] 100.0% 66.0/66.0MB downloaded
Embedding dimension 50


# Build the Embedding Matrix

Create a NumPy matrix with one row per vocabulary item and one column per embedding dimension. Use a pretrained vector when available. Use zeros for the padding row. Use zeros or a small random vector for unknown words. Do not build the vocabulary from test data.

Report the vocabulary size, embedding dimension, number of vocabulary words with pretrained vectors, and percentage covered by pretrained vectors.

In [7]:
# TODO: build embedding_matrix after you create word_to_index.
# It should have shape (vocabulary_size, word_vectors.vector_size).
# Verify that the PAD_INDEX row is zero and count pretrained hits.
# Report vocabulary size, embedding dimension, hit count, and hit percentage.
num_tokens= len(vocabulary)
embedding_dim = 50 # embedding dimmension
embedding_matrix = np.zeros((num_tokens, embedding_dim))

hit_count = 0
for i, word in enumerate(vocabulary):
    # Keras TextVectorization indices 0 (pad) and 1 (oov) are typically skipped
    # as they won't exist in the Gensim pre-trained word dictionary
    if word in word_vectors:
        embedding_matrix[i] = word_vectors[word]
        hit_count += 1
    else:
        # Assign a random normal distribution or leave as zeros for OOV tokens
        if i == 1:  # Optional: initialize the '[UNK]' token randomly
            embedding_matrix[i] = np.random.normal(size=(embedding_dim,))

embedding_layer = tf.keras.layers.Embedding(# Kera's embedding layer
     input_dim=num_tokens, #vocab size
     output_dim=embedding_dim, #dimentionality, gotta keep it consistent (50)
     weights=[embedding_matrix],
     trainable=False, #Because I am using word_vectors, I keep this false so the embeddings don't get updated by my model
     mask_zero=True # This tells the embedding layer to treat inputs with value 0 (your PAD_INDEX) as padding and to mask them out,
                    # meaning they won't contribute to the gradients or affect the output of subsequent layers.
)
#Prompt: How can I print the number of vocabulary words with pretrained vectors, and percentage covered by pretrained vectors?
# I didn't know how to do a string and a int on the same line
print(f"Vocabulary size: {num_tokens}")
print(f"Embedding dimension: {embedding_dim}")
print(f"Pretrained vector hits: {hit_count}")
hit_percentage = (hit_count / num_tokens) * 100
print(f"Percentage covered by pretrained vectors: {hit_percentage:.2f}%")

Vocabulary size: 20000
Embedding dimension: 50
Pretrained vector hits: 18361
Percentage covered by pretrained vectors: 91.81%


## Embedding layer hints

These are short examples, not complete solutions. Verify and report that the embedding layer is frozen.

In [ ]:
# PyTorch example
# embedding_layer = torch.nn.Embedding.from_pretrained(
#     embedding_matrix, freeze=True, padding_idx=PAD_INDEX
# )

# TensorFlow with Keras example
# embedding_layer = tf.keras.layers.Embedding(
#     input_dim=vocabulary_size,
#     output_dim=embedding_dimension,
#     weights=[embedding_matrix],
#     trainable=False,
#     mask_zero=True
# )

# Build and Train a Recurrent Classifier

Your model should have this structure: token indices, frozen pretrained embedding layer, RNN, GRU, or LSTM, document vector, and a four-class output layer. Begin with one recurrent layer, hidden size 64 to 128, batch size 64 to 256, three to eight epochs, and Adam. These are suggestions. Use cross-entropy classification loss. The recurrent and classification layers must be trainable.

Documentation: [PyTorch Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html), [RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html), [GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html), [LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html), [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), [Keras Embedding](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding), [SimpleRNN](https://www.tensorflow.org/api_docs/python/tf/keras/layers/SimpleRNN), [GRU](https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU), and [LSTM](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM).

In [10]:
# Choose a GRU using TensorFlow with Keras and build one recurrent classifier.
# Create training batches, train for your chosen number of epochs, and record loss per epoch.
# Use the selected device and move model and batches there when using PyTorch.
# Plot training loss by epoch.
# Report framework, device, recurrent layer, vocabulary size, model summary,
# trainable parameter count, each epoch's loss, training time, and frozen status.

#https://www.geeksforgeeks.org/machine-learning/gated-recurrent-unit-networks/
#https://medium.com/@amitkharche/rnn-lstm-gru-in-nlp-a-deep-dive-into-sequence-modeling-c241c9b8e713

#Prompts: Explain code
#Debug the 'ValueError: Argument name must be a string' error

GRU_model = Sequential([
    embedding_layer,
    GRU(64), #Set hidden state to 64, this will cause the GRU to output a 64-dimensional vector
    Dense(4, activation='softmax') # Output class=4, softmax willoutput a probability function over 4 classes (topics)
])

GRU_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
print("\nTraining the model...")
GRU_model.fit(
    train_sequences,
    train_df["label_index"],
    epochs=8, # Reduced epochs for quicker iteration, can be adjusted
    batch_size=64,
    validation_split=0.1 # Add validation split to monitor performance
)

print("Model Summary:")
GRU_model.summary()

Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 60, 50)         │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,000,000 (3.81 MB)


Training the model...
Epoch 1/8
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7911 - loss: 0.5584 - val_accuracy: 0.8820 - val_loss: 0.3431
Epoch 2/8
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8836 - loss: 0.3385 - val_accuracy: 0.8910 - val_loss: 0.3238
Epoch 3/8
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8872 - loss: 0.3182 - val_accuracy: 0.8900 - val_loss: 0.3083
Epoch 4/8
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8949 - loss: 0.3017 - val_accuracy: 0.8905 - val_loss: 0.3129
Epoch 5/8
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8993 - loss: 0.2901 - val_accuracy: 0.9000 - val_loss: 0.2973
Epoch 6/8
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9046 - loss: 0.2781 - val_accuracy: 0.8990 - val_loss: 0.2988
Epoch 7/8
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9060 - loss: 0.2657 - val_accuracy: 0.8945 - val_loss: 0.3042
Epoch 8/8
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9119 - loss: 0.2543 - 

# Task 1 - News Topic Classification

Evaluate the trained model on the official balanced test subset. Display test accuracy, a confusion matrix, five example predictions, and at least three incorrect predictions. Each prediction must include text, true label, and predicted label. Use World, Sports, Business, and Sci/Tech as category names.

In [ ]:
# TODO: evaluate on test_sequences and test_df.
# Display accuracy, a confusion matrix, five predictions, and at least three mistakes.

### Short response

Describe one pattern you noticed in the model's mistakes.

TODO

# Extract Document Vectors

Modify or reuse the model so it returns the internal document vector before the final classification layer. Suitable choices include the final RNN, GRU, or LSTM hidden state. For an LSTM, use the hidden state rather than the cell state. For a bidirectional model, you may concatenate the final forward and backward states. Extract one vector for every test document and print the final matrix shape. It should have the form number of test documents by document-vector dimension. Do not create sequence-model vectors with a second model.

In [ ]:
# TODO: encode every test document with the trained RNN.
# Store the result in test_document_vectors and print its shape.

# Task 2 - Document Similarity

Select two test documents. Encode them with the trained RNN and compare each selected vector with all test-document vectors using cosine similarity. Exclude the query document itself and display the five most similar documents for each selection.

Each result table must contain document index, category, similarity score, and text. Reference: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html

In [ ]:
# TODO: choose two test-document indices and compute cosine similarity.
# Exclude each query document from its own result and display five rows per query.

### Short response

Did the nearest documents share a category, vocabulary, or both?

TODO

# Task 3 - Query-Based Search

A new query must pass through the same tokenizer, vocabulary, unknown-word handling, sequence length, and trained RNN encoder as the documents. Encode each query and compare it with all test-document vectors using cosine similarity. Add one query of your own. Do not calculate retrieval metrics.

In [ ]:
queries = [
    "the team won the championship match",
    "technology company releases a new computer",
    "stocks rise after strong company earnings",
    "leaders meet to discuss the international conflict",
    # TODO: add one query of your own
]

# TODO: encode every query with the same pipeline used for test documents.
# For each query, display five results with rank, category, similarity score, and text.

### Short response

Which query worked best, and which query produced the weakest results?

TODO

# Final Results

Complete this compact summary table.

In [ ]:
# TODO: complete one row with your experiment results.
summary_columns = [
    "framework", "device", "recurrent layer", "hidden size",
    "number of recurrent layers", "bidirectional or unidirectional",
    "training sample size", "test sample size", "vocabulary size",
    "embedding dimension", "embedding layer frozen", "batch size",
    "number of epochs", "test accuracy", "training time"
]
summary_columns

### Final response

In a short paragraph, explain what the frozen word embeddings contributed and what the recurrent layer learned.

TODO

# Minimum required outputs

- [ ] AG News was downloaded and sampled reproducibly
- [ ] The vocabulary was built from training text only
- [ ] The embedding matrix used pretrained GloVe vectors
- [ ] The embedding layer remained frozen
- [ ] An RNN, GRU, or LSTM was trained
- [ ] Training loss was plotted
- [ ] Test accuracy and a confusion matrix were displayed
- [ ] Incorrect classifications were inspected
- [ ] RNN document vectors were extracted
- [ ] Similar documents were returned for two test documents
- [ ] Search results were returned for five text queries
- [ ] The final summary table and short responses were completed
- [ ] The notebook runs from top to bottom

Keep the assignment focused on one recurrent model in one framework. Do not add transformers, attention, language-model training, next-word prediction, trainable pretrained embeddings, sequence-to-sequence models, t-SNE, UMAP, precision at K, hyperparameter search, multiple required architectures, or a PyTorch and TensorFlow comparison.